<a href="https://colab.research.google.com/github/sanmaykant/sec/blob/main/Notebooks/data_preprocessing_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/sec-risk-analyzer"

DATA_PATH = os.path.join(BASE_PATH, "data/processed/risk_data.json")
EMBED_PATH = os.path.join(BASE_PATH, "data/embeddings")
CHUNK_PATH = os.path.join(BASE_PATH, "data/processed/chunks")

os.makedirs(EMBED_PATH, exist_ok=True)
os.makedirs(CHUNK_PATH, exist_ok=True)

print("Paths ready!")

In [ ]:
import json

with open(DATA_PATH, "r") as f:
    data = json.load(f)

print("Total filings loaded:", len(data))

In [ ]:
import re

def preprocess_text(text):
    text = re.sub(r"<.*?>", " ", text)  # remove HTML
    text = text.replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [ ]:
def chunk_text(text, min_length=100, max_length=800):
    paragraphs = re.split(r"\n{2,}", text)

    chunks = []
    buffer = ""

    for p in paragraphs:
        p = p.strip()
        if not p:
            continue

        if len(buffer) + len(p) < max_length:
            buffer += " " + p
        else:
            if len(buffer) > min_length:
                chunks.append(buffer.strip())
            buffer = p

    if len(buffer) > min_length:
        chunks.append(buffer.strip())

    return chunks

In [ ]:
from collections import defaultdict
from tqdm import tqdm

def prepare_chunks_with_metadata(results):
    all_chunks = []
    metadata = []

    year_chunks = defaultdict(list)
    company_chunks = defaultdict(list)

    for item in tqdm(results):
        ticker = item["ticker"]
        year = item["year"]

        text = preprocess_text(item["filing"])
        chunks = chunk_text(text)

        for chunk in chunks:
            all_chunks.append(chunk)

            metadata.append({
                "ticker": ticker,
                "year": year
            })

            year_chunks[year].append(chunk)
            company_chunks[ticker].append(chunk)

    return {
        "all_chunks": all_chunks,
        "metadata": metadata,
        "year_chunks": dict(year_chunks),
        "company_chunks": dict(company_chunks)
    }

In [ ]:
processed_data = prepare_chunks_with_metadata(data)

all_chunks = processed_data["all_chunks"]
metadata = processed_data["metadata"]
year_chunks = processed_data["year_chunks"]
company_chunks = processed_data["company_chunks"]

print("Total chunks:", len(all_chunks))

In [ ]:
print("\nChunks per year:")
for y in sorted(year_chunks.keys(), reverse=True):
    print(y, "→", len(year_chunks[y]))

print("\nChunks per company:")
for c in company_chunks:
    print(c, "→", len(company_chunks[c]))

In [ ]:
with open(os.path.join(CHUNK_PATH, "all_chunks.json"), "w") as f:
    json.dump({
        "chunks": all_chunks,
        "metadata": metadata
    }, f)

print("Saved flat chunks + metadata")

In [ ]:
with open(os.path.join(CHUNK_PATH, "year_chunks.json"), "w") as f:
    json.dump(year_chunks, f)

print("Saved year-wise chunks")

In [ ]:
with open(os.path.join(CHUNK_PATH, "company_chunks.json"), "w") as f:
    json.dump(company_chunks, f)

print("Saved company-wise chunks")

In [ ]:
from tqdm import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer

batch_size = 32
embeddings_list = []
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

for i in tqdm(range(0, len(all_chunks), batch_size)):
    batch = all_chunks[i:i+batch_size]
    emb = model.encode(batch, show_progress_bar=False)
    embeddings_list.append(emb)

embeddings = np.vstack(embeddings_list)

print("Embeddings shape:", embeddings.shape)

In [ ]:
import numpy as np
embed_file = os.path.join(EMBED_PATH, "all_embeddings.npy")
np.save(embed_file, embeddings)
print("Embeddings saved!")

In [ ]:
with open(os.path.join(EMBED_PATH, "metadata.json"), "w") as f:
    json.dump(metadata, f)

print("Metadata saved!")

In [ ]:
from collections import defaultdict

year_index = defaultdict(list)
company_index = defaultdict(list)

for i, m in enumerate(metadata):
    year_index[m["year"]].append(i)
    company_index[m["ticker"]].append(i)

with open(os.path.join(EMBED_PATH, "year_index.json"), "w") as f:
    json.dump(year_index, f)

with open(os.path.join(EMBED_PATH, "company_index.json"), "w") as f:
    json.dump(company_index, f)

print("Index mappings saved!")

In [ ]:
print("Sample chunk:\n", all_chunks[0][:300])
print("\nEmbedding sample:\n", embeddings[0][:10])
print("\nMetadata sample:\n", metadata[0])